## Self-contained setup (Colab-safe)

This notebook needs `anchored_sgld.py`, which normally sits next to it in the
repository. The cell below **writes that module to the working directory**, so
the notebook runs anywhere — Colab, a fresh Jupyter kernel, a bare container —
with no file upload and no network access.

The content is the library verbatim; if you already have the file next to the
notebook, this simply rewrites the identical text. Everything Colab needs beyond
it (NumPy, SciPy, pandas, scikit-learn, Matplotlib) is preinstalled, so there is
nothing to `pip install`.

In [ ]:
%%writefile anchored_sgld.py
"""Non-reversible anchored Langevin with block state-dependent skew-symmetric J.

Reusable library behind ``nonreversible_anchored_langevin.ipynb``.

The sampled object is the regression coefficient ``beta`` (written ``b`` in
code where it is a projection argument).  It is *unknown* and is what the
sampler explores; ``beta_true`` is the fixed vector used once to generate the
labels and is never used by the sampler.

Target
------
Uniform prior on a constraint set ``K`` times the logistic likelihood:

    pi_K(beta) ∝ exp(-U(beta)) 1_K(beta),
    U(beta) = sum_{j in train} [ log(1 + exp(X_j.beta)) - y_j X_j.beta ].

``U`` is a **sum** over the 1600 training rows, never a mean.  The mini-batch
estimator rescales accordingly,

    Ghat_k = (n_train / m) X_{B_k}^T [ sigmoid(X_{B_k} beta_k) - y_{B_k} ],

with ``B_k`` drawn uniformly without replacement within the iteration.

Anchor
------
    U0(beta) = U(beta) + rho H_K(beta),
    a(beta)  = exp(U(beta) - U0(beta)) = exp(-rho H_K(beta)).

``H_K`` is a purely geometric function of the constraint (``||beta||^2`` on the
ball, the normalised constraint value on the quartic set), so ``a`` is computed
**exactly from the geometry** — a noisy likelihood difference is never
exponentiated.  With ``rho = log 2`` and ``H_K in [0, 1]`` on ``K`` this gives
``1/2 <= a <= 1``.

This anchor is a *proposed non-trivial anchor for an already smooth target*,
used to make ``a`` state-dependent so the anchored machinery is exercised.  It
is **not** an extra Bayesian penalty: the target ``pi_K`` is unchanged, because
``a`` enters both the drift and the diffusion coefficient in the combination
that leaves ``exp(-U)`` invariant (see :func:`invariant_measure_note`).

Update
------
    beta_{k+1} = Pi_K[ beta_k - h a_k (v_k + alpha J(beta_k) v_k)
                       + sqrt(2 h a_k) xi_k ],
    v_k = Ghat_k + rho grad_H_K(beta_k),
    a_k = exp(-rho H_K(beta_k)),  xi_k ~ N(0, I_d).

No ``grad a`` correction is added, and ``J`` is never rescaled beyond the
constant block strengths ``s``.
"""

from __future__ import annotations

import math
import time
from dataclasses import dataclass, field, replace
from typing import Callable, Sequence

import numpy as np
from scipy.optimize import brentq
from scipy.special import expit
from sklearn.model_selection import train_test_split

RHO_ANCHORED: float = math.log(2.0)

BETA_TRUE_9 = np.array(
    [0.35, -0.25, 0.15, 0.30, -0.20, 0.10, 0.25, -0.15, 0.20]
)
BETA_TRUE_3 = np.array([0.60, -0.30, 0.20])


# ==========================================================================
# 1. Configuration
# ==========================================================================
@dataclass(frozen=True)
class ExperimentConfig:
    """Every knob of the experiment.  Nothing is hard-coded elsewhere."""

    d: int = 9
    n_total: int = 2000
    test_fraction: float = 0.2
    batch_size: int = 50                       # m
    n_iterations: int = 1000
    step_size: float = 1e-4                    # h (the paper's candidate value)
    n_repeats: int = 100                       # R
    block_scales: tuple[float, ...] = (10.0, 10.0, 10.0)
    epsilon: float = 0.2                       # smoothing of the l^p constraints
    Lambda: float = 1.0                        # quartic threshold
    l1_radius: float = 3.0                     # L1-smooth ball radius budget
    checkpoint_every: int = 10
    data_seed: int = 2026
    split_seed: int = 2027
    sampler_seed: int = 3000
    # Step-size sensitivity
    sensitivity_divisors: tuple[float, ...] = (1.0, 2.0, 4.0)
    sensitivity_repeats: int = 20
    # Reporting
    target_accuracy: float = 0.64

    @property
    def n_train(self) -> int:
        return self.n_total - int(round(self.n_total * self.test_fraction))

    @property
    def n_test(self) -> int:
        return int(round(self.n_total * self.test_fraction))

    @property
    def n_blocks(self) -> int:
        if self.d % 3 != 0:
            raise ValueError("d must be a multiple of 3 for the block construction")
        return self.d // 3

    @property
    def scales(self) -> np.ndarray:
        """Block strengths as an array of length ``n_blocks``."""
        s = np.asarray(self.block_scales, dtype=float)
        if s.size != self.n_blocks:
            raise ValueError(
                f"block_scales has {s.size} entries but d = {self.d} needs "
                f"{self.n_blocks}"
            )
        return s

    def beta_true(self) -> np.ndarray:
        if self.d == 9:
            return BETA_TRUE_9.copy()
        if self.d == 3:
            return BETA_TRUE_3.copy()
        raise ValueError("beta_true is specified only for d = 3 and d = 9")


CONFIG_D3 = ExperimentConfig(d=3, block_scales=(10.0,))


#: The four compared methods: (name, rho, alpha).
METHODS: tuple[tuple[str, float, float], ...] = (
    ("Projected SGLD", 0.0, 0.0),
    ("Non-reversible SGLD", 0.0, 1.0),
    ("Reversible anchored Langevin", RHO_ANCHORED, 0.0),
    ("Non-reversible anchored Langevin", RHO_ANCHORED, 1.0),
)

#: Colourblind-safe, validated (worst adjacent CVD deltaE 9.2 protan / 22.9
#: normal).  Line style is a deliberate second encoding channel.
METHOD_STYLE: dict[str, dict[str, object]] = {
    "Projected SGLD": {"color": "#0173B2", "ls": "-", "lw": 1.7},
    "Non-reversible SGLD": {"color": "#DE8F05", "ls": "--", "lw": 1.7},
    "Reversible anchored Langevin": {"color": "#029E73", "ls": "-.", "lw": 1.7},
    "Non-reversible anchored Langevin": {"color": "#CC3311", "ls": "-", "lw": 2.8},
}


def invariant_measure_note() -> str:
    """The continuous-time identity, and what it does *not* claim."""
    return (
        "Continuous process:  d.beta = -a(beta) [I + alpha J(beta)] grad_U0(beta) dt "
        "+ sqrt(2 a(beta)) dW.\n"
        "  * Reversible part.  For pi ∝ exp(-U0)/a the Fokker-Planck flux is\n"
        "    -b pi + grad(a pi) = a grad_U0 pi - grad_U0 (a pi) = 0, since a pi ∝ "
        "exp(-U0).\n"
        "    With a = exp(U - U0) this gives pi ∝ exp(-U0)/a = exp(-U): the anchor "
        "cancels exactly.\n"
        "  * Non-reversible part.  Its flux is -alpha a J grad_U0 pi = "
        "alpha J grad(exp(-U0)) x const,\n"
        "    whose divergence is (div J).grad f + trace(J Hess f) = 0 + 0, because "
        "div J = 0 and a\n"
        "    skew matrix has zero Frobenius inner product with a symmetric one. So "
        "pi is unchanged.\n"
        "\n"
        "WHAT THIS DOES NOT SAY.  The implemented algorithm is a *projected "
        "stochastic-gradient*\n"
        "Euler-Maruyama discretisation. Three separate sources of bias remain, none "
        "of which the\n"
        "identity above controls:\n"
        "  (i)   finite step size h (Euler-Maruyama discretisation bias, O(h) in "
        "general);\n"
        "  (ii)  mini-batch gradient noise, which injects extra variance not matched "
        "by the\n"
        "        sqrt(2 h a) term and inflates the effective temperature;\n"
        "  (iii) the projection Pi_K, which puts mass on the boundary and is not a "
        "discretisation\n"
        "        of any reflected process that preserves pi_K exactly.\n"
        "Accuracy curves therefore say nothing directly about posterior fidelity."
    )


# ==========================================================================
# 2. Synthetic data
# ==========================================================================
@dataclass
class Dataset:
    """Frozen data set and stratified split, shared by every method/geometry."""

    X_train: np.ndarray
    y_train: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    beta_true: np.ndarray
    data_seed: int
    split_seed: int

    @property
    def n_train(self) -> int:
        return self.X_train.shape[0]

    @property
    def n_test(self) -> int:
        return self.X_test.shape[0]

    @property
    def d(self) -> int:
        return self.X_train.shape[1]


def make_dataset(cfg: ExperimentConfig) -> Dataset:
    """Generate ``X_j ~ N(0, 2 I_d)`` and Bernoulli labels, then split 80/20.

    Coordinate standard deviation is ``sqrt(2)``.  There is no intercept and no
    feature standardisation.  Labels use the inverse-CDF form
    ``y_j = 1{u_j <= sigmoid(X_j.beta_true)}`` with ``u_j ~ Uniform(0,1)``.
    """
    rng = np.random.default_rng(cfg.data_seed)
    beta_true = cfg.beta_true()
    X = rng.normal(loc=0.0, scale=np.sqrt(2.0), size=(cfg.n_total, cfg.d))
    u = rng.uniform(0.0, 1.0, size=cfg.n_total)
    y = (u <= expit(X @ beta_true)).astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=cfg.test_fraction,
        random_state=cfg.split_seed,
        stratify=y,
        shuffle=True,
    )
    return Dataset(
        X_train=np.ascontiguousarray(X_train),
        y_train=np.ascontiguousarray(y_train),
        X_test=np.ascontiguousarray(X_test),
        y_test=np.ascontiguousarray(y_test),
        beta_true=beta_true,
        data_seed=cfg.data_seed,
        split_seed=cfg.split_seed,
    )


# ==========================================================================
# 3. Posterior and gradients  (beta may be (d,) or (R, d))
# ==========================================================================
def _as_2d(beta: np.ndarray) -> tuple[np.ndarray, bool]:
    beta = np.asarray(beta, dtype=float)
    if beta.ndim == 1:
        return beta[None, :], True
    return beta, False


def potential_U(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """``U(beta) = sum_j [softplus(X_j.beta) - y_j X_j.beta]`` (a SUM).

    ``np.logaddexp(0, z)`` is the numerically stable softplus; the
    label-dependent term ``- y_j X_j.beta`` is retained.
    """
    beta2, squeeze = _as_2d(beta)
    eta = beta2 @ X.T
    value = (np.logaddexp(0.0, eta) - y[None, :] * eta).sum(axis=1)
    return value[0] if squeeze else value


def full_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """``grad U = X^T (sigmoid(X beta) - y)`` over all rows of ``X``."""
    beta2, squeeze = _as_2d(beta)
    residual = expit(beta2 @ X.T) - y[None, :]
    gradient = residual @ X
    return gradient[0] if squeeze else gradient


def minibatch_gradient(
    beta: np.ndarray,
    X: np.ndarray,
    y: np.ndarray,
    batch_index: np.ndarray,
) -> np.ndarray:
    """``Ghat = (n_train/m) X_B^T [sigmoid(X_B beta) - y_B]``, vectorised over replicates.

    Parameters
    ----------
    beta
        ``(R, d)`` current states.
    X, y
        Full training arrays; ``n_train`` is taken from ``X``.
    batch_index
        ``(R, m)`` integer indices, drawn uniformly **without replacement**
        within each row.

    The ``n_train / m`` factor makes this an unbiased estimator of the *summed*
    gradient.  It is never replaced by an unscaled batch average, and never uses
    ``n_total``.
    """
    beta2, squeeze = _as_2d(beta)
    n_train, m = X.shape[0], batch_index.shape[-1]
    Xb = X[batch_index]                       # (R, m, d)
    yb = y[batch_index]                       # (R, m)
    eta = np.einsum("rmd,rd->rm", Xb, beta2)
    residual = expit(eta) - yb
    gradient = (n_train / m) * np.einsum("rmd,rm->rd", Xb, residual)
    return gradient[0] if squeeze else gradient


def accuracy(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Plug-in accuracy of ``1{sigmoid(X.beta) >= 0.5}``, vectorised over replicates."""
    beta2, squeeze = _as_2d(beta)
    prediction = (beta2 @ X.T) >= 0.0        # sigmoid(z) >= 0.5  <=>  z >= 0
    value = (prediction == (y[None, :] >= 0.5)).mean(axis=1)
    return value[0] if squeeze else value


# ==========================================================================
# 4. Geometries: constraint, anchor, projection, initialisation, J vectors
# ==========================================================================
class Geometry:
    """Interface shared by the ball and the quartic constraint set."""

    name: str
    d: int

    # --- constraint ---
    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    @property
    def threshold(self) -> float:
        raise NotImplementedError

    def feasible(self, beta: np.ndarray, tol: float = 1e-9) -> np.ndarray:
        return self.constraint_value(beta) <= self.threshold + tol

    # --- anchor ---
    def H(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    # --- geometry of the boundary ---
    def normal(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    # --- block vectors defining J (before the block strengths) ---
    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``(R, n_blocks, 3)`` vectors ``w_l`` with block ``[s_l w_l]_x``."""
        raise NotImplementedError

    # --- projection and initialisation ---
    def project(self, z: np.ndarray) -> "ProjectionOutcome":
        raise NotImplementedError

    def sample_uniform(self, rng: np.random.Generator, n: int) -> np.ndarray:
        raise NotImplementedError


@dataclass
class ProjectionOutcome:
    """Result of projecting a batch of proposals."""

    beta: np.ndarray                 # (R, d) projected states
    projected: np.ndarray            # (R,) bool, was the row infeasible?
    max_kkt_residual: float          # worst stationarity residual over projected rows
    max_feasibility_excess: float    # worst g(b) - threshold after projection


class BallGeometry(Geometry):
    """``K = {beta : ||beta||_2^2 <= 1}``, with ``H_K(beta) = ||beta||^2``."""

    name = "unit ball"

    def __init__(self, d: int) -> None:
        self.d = d

    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        value = np.sum(beta2 * beta2, axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return 1.0

    def H(self, beta: np.ndarray) -> np.ndarray:
        return self.constraint_value(beta)

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        return 2.0 * np.asarray(beta, dtype=float)

    def normal(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        norm = np.linalg.norm(beta2, axis=1, keepdims=True)
        out = np.divide(beta2, norm, out=np.zeros_like(beta2), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        return direction / np.linalg.norm(direction)

    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``w_l = beta_{I_l}`` — the ball's normal is proportional to ``beta``."""
        beta2, _ = _as_2d(beta)
        return beta2.reshape(beta2.shape[0], -1, 3)

    def project(self, z: np.ndarray) -> ProjectionOutcome:
        """Euclidean projection: ``z`` if ``||z|| <= 1``, else ``z/||z||``."""
        z2, squeeze = _as_2d(z)
        norm = np.linalg.norm(z2, axis=1)
        outside = norm > 1.0
        beta = z2.copy()
        if np.any(outside):
            beta[outside] = z2[outside] / norm[outside, None]
        # KKT: b(1 + 2 mu) = z with mu = (||z|| - 1)/2; residual is exactly 0.
        residual = 0.0
        if np.any(outside):
            mu = (norm[outside] - 1.0) / 2.0
            residual = float(
                np.abs(beta[outside] * (1.0 + 2.0 * mu[:, None]) - z2[outside]).max()
            )
        excess = float((np.sum(beta * beta, axis=1) - 1.0).max())
        out = beta[0] if squeeze else beta
        return ProjectionOutcome(out, outside, residual, excess)

    def sample_uniform(self, rng: np.random.Generator, n: int) -> np.ndarray:
        """Uniform on the ball: ``Z V^{1/d} / ||Z||``."""
        Z = rng.standard_normal((n, self.d))
        V = rng.random(n)
        radius = V ** (1.0 / self.d)
        return Z * (radius / np.linalg.norm(Z, axis=1))[:, None]


class QuarticGeometry(Geometry):
    """``K = {beta : g(beta) = sum_i (beta_i^2 + eps^2)^2 <= Lambda}``.

    ``g_min = d eps^4`` is attained at the origin, ``D = Lambda - d eps^4``, and

        H_K(beta) = (g(beta) - d eps^4) / D  in [0, 1] on K,
        grad_g[i] = 4 beta_i (beta_i^2 + eps^2),  grad_H_K = grad_g / D.
    """

    name = "quartic set"

    def __init__(self, d: int, epsilon: float = 0.2, Lambda: float = 1.0) -> None:
        self.d = d
        self.epsilon = epsilon
        self.Lambda = Lambda
        self.g_min = d * epsilon ** 4
        self.D = Lambda - self.g_min
        if self.D <= 0:
            raise ValueError("Lambda must exceed d * epsilon**4")

    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        value = np.sum((beta2 * beta2 + self.epsilon ** 2) ** 2, axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return self.Lambda

    def grad_g(self, beta: np.ndarray) -> np.ndarray:
        beta = np.asarray(beta, dtype=float)
        return 4.0 * beta * (beta * beta + self.epsilon ** 2)

    def H(self, beta: np.ndarray) -> np.ndarray:
        return (self.constraint_value(beta) - self.g_min) / self.D

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        return self.grad_g(beta) / self.D

    def normal(self, beta: np.ndarray) -> np.ndarray:
        gradient, squeeze = _as_2d(self.grad_g(beta))
        norm = np.linalg.norm(gradient, axis=1, keepdims=True)
        out = np.divide(gradient, norm, out=np.zeros_like(gradient), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        objective = lambda t: float(self.constraint_value(t * direction)) - self.Lambda
        upper = 1.0
        while objective(upper) < 0.0:
            upper *= 2.0
        t = brentq(objective, 0.0, upper, xtol=1e-14, rtol=8.9e-16, maxiter=200)
        return t * direction

    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``w_l = -grad_{I_l} g(beta)``.

        The quartic normal is proportional to ``grad g``, not to ``beta``, so
        the ball's ``[s beta]_x`` block would generally violate ``J n = 0`` here.
        """
        gradient, _ = _as_2d(self.grad_g(beta))
        return (-gradient).reshape(gradient.shape[0], -1, 3)

    # ---- Euclidean projection via the KKT system ----
    def _solve_coordinates(self, z_abs: np.ndarray, mu) -> np.ndarray:
        """Solve ``b + 4 mu b (b^2 + eps^2) = |z|`` for ``b >= 0``, elementwise.

        The equation is the depressed cubic ``b^3 + p b + q = 0`` with
        ``p = (1 + 4 mu eps^2)/(4 mu) > 0`` and ``q = -|z|/(4 mu)``.  With
        ``p > 0`` there is exactly one real root, given stably by the hyperbolic
        form ``b = -2 sqrt(p/3) sinh( arcsinh( q / (2 (p/3)^{3/2}) ) / 3 )``.
        The map is strictly increasing in ``b`` (derivative
        ``1 + 12 mu b^2 + 4 mu eps^2 > 0``), so the root is unique.
        """
        mu_array = np.asarray(mu, dtype=float)
        scalar_mu = mu_array.ndim == 0
        if scalar_mu and mu_array <= 0.0:
            return z_abs.copy()
        safe = np.where(mu_array > 0.0, mu_array, 1.0)
        p = (1.0 + 4.0 * safe * self.epsilon ** 2) / (4.0 * safe)
        q = -z_abs / (4.0 * safe)
        scale = (p / 3.0) ** 1.5
        theta = np.arcsinh(q / (2.0 * scale)) / 3.0
        root = -2.0 * np.sqrt(p / 3.0) * np.sinh(theta)
        # mu = 0 leaves the point unchanged (the map is the identity there).
        return np.where(np.broadcast_to(mu_array > 0.0, root.shape), root, z_abs)

    def _project_one(self, z: np.ndarray) -> tuple[np.ndarray, float]:
        """Project a single infeasible point; returns ``(b, mu)``."""
        sign = np.sign(z)
        z_abs = np.abs(z)

        def gap(mu: float) -> float:
            b = self._solve_coordinates(z_abs, mu)
            return float(np.sum((b * b + self.epsilon ** 2) ** 2)) - self.Lambda

        # gap(0) = g(z) - Lambda > 0; gap is decreasing and tends to
        # d eps^4 - Lambda < 0, so a bracket always exists.
        mu_high = 1.0
        for _ in range(200):
            if gap(mu_high) <= 0.0:
                break
            mu_high *= 2.0
        else:  # pragma: no cover
            raise RuntimeError("failed to bracket the projection multiplier mu")
        mu = brentq(gap, 0.0, mu_high, xtol=1e-14, rtol=8.9e-16, maxiter=300)
        return sign * self._solve_coordinates(z_abs, mu), float(mu)

    def project(self, z: np.ndarray, n_bisect: int = 100) -> ProjectionOutcome:
        """Exact Euclidean projection onto ``{g <= Lambda}`` (never radial scaling).

        The outer multiplier ``mu`` is found by a bracketed bisection run on all
        infeasible rows simultaneously; ``mu -> g(b(mu))`` is monotonically
        decreasing, so bisection is unconditionally reliable, and ``n_bisect``
        halvings of the bracket reach machine precision.  :meth:`_project_one`
        keeps the scalar ``brentq`` version, which the checks use as an
        independent reference.
        """
        z2, squeeze = _as_2d(z)
        beta = z2.copy()
        outside = self.constraint_value(z2) > self.Lambda
        worst_kkt = 0.0

        rows = np.nonzero(outside)[0]
        if rows.size:
            z_rows = z2[rows]
            sign, z_abs = np.sign(z_rows), np.abs(z_rows)

            def gap(mu_column: np.ndarray) -> np.ndarray:
                b = self._solve_coordinates(z_abs, mu_column)
                return np.sum((b * b + self.epsilon ** 2) ** 2, axis=1) - self.Lambda

            # gap(0) > 0 by construction; grow the upper end until gap <= 0.
            low = np.zeros(rows.size)
            high = np.ones(rows.size)
            for _ in range(200):
                need = gap(high[:, None]) > 0.0
                if not need.any():
                    break
                high[need] *= 2.0
            else:  # pragma: no cover
                raise RuntimeError("failed to bracket the projection multiplier mu")

            for _ in range(n_bisect):
                mid = 0.5 * (low + high)
                positive = gap(mid[:, None]) > 0.0
                low = np.where(positive, mid, low)
                high = np.where(positive, high, mid)
            mu = 0.5 * (low + high)

            b = sign * self._solve_coordinates(z_abs, mu[:, None])
            beta[rows] = b
            worst_kkt = float(
                np.abs(b + mu[:, None] * self.grad_g(b) - z_rows).max()
            )
        excess = float((self.constraint_value(beta) - self.Lambda).max())
        out = beta[0] if squeeze else beta
        return ProjectionOutcome(out, outside, worst_kkt, excess)

    @property
    def box_half_width(self) -> float:
        """``b = sqrt( sqrt(Lambda - (d-1) eps^4) - eps^2 )``."""
        inner = math.sqrt(self.Lambda - (self.d - 1) * self.epsilon ** 4)
        return math.sqrt(inner - self.epsilon ** 2)

    def sample_uniform(
        self, rng: np.random.Generator, n: int, max_rounds: int = 10_000
    ) -> np.ndarray:
        """Uniform on ``K`` by rejection sampling from ``[-b, b]^d``."""
        half = self.box_half_width
        accepted: list[np.ndarray] = []
        total = 0
        kept = 0
        for _ in range(max_rounds):
            proposals = rng.uniform(-half, half, size=(max(n, 256), self.d))
            total += proposals.shape[0]
            good = proposals[self.constraint_value(proposals) <= self.Lambda]
            kept += good.shape[0]
            if good.size:
                accepted.append(good)
            if sum(a.shape[0] for a in accepted) >= n:
                break
        else:  # pragma: no cover
            raise RuntimeError("rejection sampler failed to fill the requested draws")
        out = np.vstack(accepted)[:n]
        self.last_acceptance_rate = kept / total
        return out


def make_geometry(name: str, cfg: ExperimentConfig) -> Geometry:
    if name == "ball":
        return BallGeometry(cfg.d)
    if name == "quartic":
        return QuarticGeometry(cfg.d, cfg.epsilon, cfg.Lambda)
    if name in ("l1smooth", "l1_smooth_ball"):
        return L1SmoothBallGeometry(cfg.d, cfg.epsilon, cfg.l1_radius)
    raise ValueError(f"unknown geometry {name!r}")


# ==========================================================================
# 5. Block state-dependent skew-symmetric matrix
# ==========================================================================
def apply_J(
    beta: np.ndarray,
    v: np.ndarray,
    geometry: Geometry,
    scales: np.ndarray,
) -> np.ndarray:
    """Matrix-free ``J(beta) v`` via block cross products.

    Each block acts as ``[s_l w_l]_x v_{I_l} = (s_l w_l) x v_{I_l}``, so no
    ``d x d`` matrix is ever formed.  The block strength multiplies ``w_l``
    exactly once.
    """
    beta2, squeeze = _as_2d(beta)
    v2, _ = _as_2d(v)
    w = geometry.j_block_vectors(beta2) * np.asarray(scales)[None, :, None]
    blocks = np.cross(w, v2.reshape(v2.shape[0], -1, 3))
    out = blocks.reshape(v2.shape[0], -1)
    return out[0] if squeeze else out


def hat(w: np.ndarray) -> np.ndarray:
    """``[w]_x``: the 3x3 cross-product matrix with ``[w]_x v = w x v``."""
    w1, w2, w3 = float(w[0]), float(w[1]), float(w[2])
    return np.array(
        [
            [0.0, -w3, w2],
            [w3, 0.0, -w1],
            [-w2, w1, 0.0],
        ]
    )


def build_J(beta: np.ndarray, geometry: Geometry, scales: np.ndarray) -> np.ndarray:
    """Explicit ``d x d`` matrix ``J(beta)`` — for verification, not for sampling."""
    beta = np.asarray(beta, dtype=float).ravel()
    d = beta.size
    w = geometry.j_block_vectors(beta[None, :])[0] * np.asarray(scales)[:, None]
    J = np.zeros((d, d))
    for block, vector in enumerate(w):
        lo = 3 * block
        J[lo : lo + 3, lo : lo + 3] = hat(vector)
    return J


def divergence_J(
    beta: np.ndarray, geometry: Geometry, scales: np.ndarray, step: float = 1e-5
) -> np.ndarray:
    """Centred finite-difference ``div(J)_i = sum_j dJ_ij/dbeta_j``."""
    beta = np.asarray(beta, dtype=float).ravel()
    divergence = np.zeros(beta.size)
    for j in range(beta.size):
        plus, minus = beta.copy(), beta.copy()
        plus[j] += step
        minus[j] -= step
        divergence += (
            build_J(plus, geometry, scales)[:, j]
            - build_J(minus, geometry, scales)[:, j]
        ) / (2.0 * step)
    return divergence


# ==========================================================================
# 6. Random streams (shared across the four methods)
# ==========================================================================
GEOMETRY_ID: dict[str, int] = {"unit ball": 11, "quartic set": 22,
                               "L1-smooth ball": 33}


@dataclass
class Streams:
    """Pre-generated randomness, identical for all four methods.

    Within one replicate and geometry every method sees the same starting
    coefficients, the same mini-batch index sequence and the same Gaussian
    increments; replicates use independent sub-streams.  Batch indices and
    Gaussian increments come from **separate** spawned streams, so the noise is
    independent of the current mini-batch.
    """

    beta_init: np.ndarray            # (R, d)
    batch_index: np.ndarray          # (R, n_iterations, m) int32
    noise: np.ndarray                # (R, n_iterations, d)
    seed_key: tuple[int, ...]

    @property
    def n_repeats(self) -> int:
        return self.beta_init.shape[0]

    @property
    def n_iterations(self) -> int:
        return self.noise.shape[1]


def make_streams(
    cfg: ExperimentConfig,
    geometry: Geometry,
    n_iterations: int,
    n_repeats: int,
    n_train: int,
) -> Streams:
    """Build the shared randomness for one geometry."""
    key = (cfg.sampler_seed, GEOMETRY_ID[geometry.name])
    base = np.random.SeedSequence(list(key))
    init_ss, batch_ss, noise_ss = base.spawn(3)

    beta_init = geometry.sample_uniform(np.random.default_rng(init_ss), n_repeats)

    batch_index = np.empty((n_repeats, n_iterations, cfg.batch_size), dtype=np.int32)
    for r, seed in enumerate(batch_ss.spawn(n_repeats)):
        rng = np.random.default_rng(seed)
        for k in range(n_iterations):
            batch_index[r, k] = rng.choice(n_train, size=cfg.batch_size, replace=False)

    noise = np.empty((n_repeats, n_iterations, cfg.d))
    for r, seed in enumerate(noise_ss.spawn(n_repeats)):
        noise[r] = np.random.default_rng(seed).standard_normal((n_iterations, cfg.d))

    return Streams(beta_init, batch_index, noise, key)


# ==========================================================================
# 7. The common sampler
# ==========================================================================
@dataclass
class RunResult:
    """Checkpointed output of one (method, geometry) run over ``R`` replicates."""

    method: str
    geometry: str
    rho: float
    alpha: float
    step_size: float
    n_iterations: int
    n_repeats: int
    block_scales: tuple[float, ...]
    checkpoints: np.ndarray          # (n_ckpt,) iteration indices
    train_accuracy: np.ndarray       # (n_ckpt, R)
    test_accuracy: np.ndarray        # (n_ckpt, R)
    beta: np.ndarray                 # (n_ckpt, R, d)
    train_loss: np.ndarray           # (n_ckpt, R)  full U on the training set
    constraint: np.ndarray           # (n_ckpt, R)  ||beta||^2 or g(beta)
    anchor: np.ndarray               # (n_ckpt, R)  a(beta)
    projection_rate: float
    max_kkt_residual: float
    max_feasibility_excess: float
    n_nonfinite: int
    runtime: float
    full_gradient: bool = False
    #: Mean over iterations and replicates of ||alpha J v|| / ||v||: how much
    #: larger the added non-reversible term is than the reversible drift.
    drift_ratio: float = 0.0
    #: Mean per-step displacement ||proposal - beta|| before projection.
    step_displacement: float = 0.0

    @property
    def simulated_time(self) -> np.ndarray:
        """``t = k h`` — the axis on which different step sizes are comparable."""
        return self.checkpoints * self.step_size

    def mean_std(self, which: str = "test_accuracy") -> tuple[np.ndarray, np.ndarray]:
        """Across-replicate mean and sample standard deviation (``ddof=1``)."""
        values = getattr(self, which)
        return values.mean(axis=1), values.std(axis=1, ddof=1)


def run_sampler(
    dataset: Dataset,
    geometry: Geometry,
    streams: Streams,
    *,
    method: str,
    rho: float,
    alpha: float,
    scales: np.ndarray,
    step_size: float,
    n_iterations: int,
    checkpoint_every: int,
    use_full_gradient: bool = False,
) -> RunResult:
    """One implementation; the four methods differ only in ``rho`` and ``alpha``.

        beta_{k+1} = Pi_K[ beta_k - h a_k (v_k + alpha J(beta_k) v_k)
                           + sqrt(2 h a_k) xi_k ]

    No ``grad a`` correction is added and ``J`` is never rescaled.
    """
    X_train, y_train = dataset.X_train, dataset.y_train
    X_test, y_test = dataset.X_test, dataset.y_test
    h = step_size
    beta = streams.beta_init.copy()
    n_repeats = beta.shape[0]

    checkpoints = [0]
    train_accuracy = [accuracy(beta, X_train, y_train)]
    test_accuracy = [accuracy(beta, X_test, y_test)]
    beta_history = [beta.copy()]
    train_loss = [potential_U(beta, X_train, y_train)]
    constraint = [geometry.constraint_value(beta)]
    anchor = [np.exp(-rho * geometry.H(beta))]

    n_projected = 0
    n_nonfinite = 0
    worst_kkt = 0.0
    worst_excess = -np.inf
    ratio_sum = 0.0
    displacement_sum = 0.0

    start = time.perf_counter()
    for k in range(n_iterations):
        if use_full_gradient:
            gradient = full_gradient(beta, X_train, y_train)
        else:
            gradient = minibatch_gradient(
                beta, X_train, y_train, streams.batch_index[:, k, :]
            )

        # v = Ghat + rho grad_H: the anchor derivative is added ONCE and is not
        # multiplied by n_train/m.
        v = gradient if rho == 0.0 else gradient + rho * geometry.grad_H(beta)
        a = np.ones(n_repeats) if rho == 0.0 else np.exp(-rho * geometry.H(beta))

        if alpha == 0.0:
            drift = v
        else:
            non_reversible = alpha * apply_J(beta, v, geometry, scales)
            drift = v + non_reversible
            # Diagnostic: how big is the added term relative to the reversible one?
            ratio_sum += float(
                np.mean(
                    np.linalg.norm(non_reversible, axis=1)
                    / np.maximum(np.linalg.norm(v, axis=1), 1e-300)
                )
            )
        proposal = (
            beta
            - h * a[:, None] * drift
            + np.sqrt(2.0 * h * a)[:, None] * streams.noise[:, k, :]
        )

        displacement_sum += float(np.mean(np.linalg.norm(proposal - beta, axis=1)))

        bad = ~np.isfinite(proposal).all(axis=1)
        if np.any(bad):
            n_nonfinite += int(bad.sum())
            proposal[bad] = beta[bad]          # freeze; counted and reported

        outcome = geometry.project(proposal)
        beta = outcome.beta
        n_projected += int(outcome.projected.sum())
        worst_kkt = max(worst_kkt, outcome.max_kkt_residual)
        worst_excess = max(worst_excess, outcome.max_feasibility_excess)

        if (k + 1) % checkpoint_every == 0:
            checkpoints.append(k + 1)
            train_accuracy.append(accuracy(beta, X_train, y_train))
            test_accuracy.append(accuracy(beta, X_test, y_test))
            beta_history.append(beta.copy())
            train_loss.append(potential_U(beta, X_train, y_train))
            constraint.append(geometry.constraint_value(beta))
            anchor.append(np.exp(-rho * geometry.H(beta)))
    runtime = time.perf_counter() - start

    return RunResult(
        method=method,
        geometry=geometry.name,
        rho=rho,
        alpha=alpha,
        step_size=h,
        n_iterations=n_iterations,
        n_repeats=n_repeats,
        block_scales=tuple(float(x) for x in np.atleast_1d(scales)),
        checkpoints=np.asarray(checkpoints),
        train_accuracy=np.asarray(train_accuracy),
        test_accuracy=np.asarray(test_accuracy),
        beta=np.asarray(beta_history),
        train_loss=np.asarray(train_loss),
        constraint=np.asarray(constraint),
        anchor=np.asarray(anchor),
        projection_rate=n_projected / (n_iterations * n_repeats),
        max_kkt_residual=worst_kkt,
        max_feasibility_excess=float(worst_excess),
        n_nonfinite=n_nonfinite,
        runtime=runtime,
        full_gradient=use_full_gradient,
        drift_ratio=ratio_sum / n_iterations,
        step_displacement=displacement_sum / n_iterations,
    )


def run_all_methods(
    dataset: Dataset,
    geometry: Geometry,
    cfg: ExperimentConfig,
    *,
    n_iterations: int | None = None,
    n_repeats: int | None = None,
    step_size: float | None = None,
    checkpoint_every: int | None = None,
    use_full_gradient: bool = False,
    verbose: bool = True,
) -> dict[str, RunResult]:
    """Run all four methods on one geometry with shared randomness."""
    n_iterations = n_iterations or cfg.n_iterations
    n_repeats = n_repeats or cfg.n_repeats
    step_size = cfg.step_size if step_size is None else step_size
    checkpoint_every = checkpoint_every or cfg.checkpoint_every

    streams = make_streams(cfg, geometry, n_iterations, n_repeats, dataset.n_train)
    results: dict[str, RunResult] = {}
    for name, rho, alpha in METHODS:
        result = run_sampler(
            dataset, geometry, streams,
            method=name, rho=rho, alpha=alpha, scales=cfg.scales,
            step_size=step_size, n_iterations=n_iterations,
            checkpoint_every=checkpoint_every, use_full_gradient=use_full_gradient,
        )
        results[name] = result
        if verbose:
            mean, sd = result.mean_std("test_accuracy")
            print(
                f"    {name:<34} final test acc {mean[-1]:.4f} +/- {sd[-1]:.4f}  "
                f"proj rate {result.projection_rate:.4f}  "
                f"nonfinite {result.n_nonfinite}  {result.runtime:5.1f}s",
                flush=True,
            )
    return results


# ==========================================================================
# 8. Mathematical implementation checks
# ==========================================================================
def check_gradient_finite_difference(
    dataset: Dataset, seed: int = 7, step: float = 1e-6
) -> dict[str, float]:
    """Compare ``full_gradient`` with centred finite differences of ``U``."""
    rng = np.random.default_rng(seed)
    beta = rng.normal(scale=0.3, size=dataset.d)
    analytic = full_gradient(beta, dataset.X_train, dataset.y_train)
    numerical = np.zeros_like(beta)
    for i in range(beta.size):
        plus, minus = beta.copy(), beta.copy()
        plus[i] += step
        minus[i] -= step
        numerical[i] = (
            potential_U(plus, dataset.X_train, dataset.y_train)
            - potential_U(minus, dataset.X_train, dataset.y_train)
        ) / (2.0 * step)
    absolute = float(np.abs(analytic - numerical).max())
    return {
        "max_absolute_error": absolute,
        "max_relative_error": absolute / float(np.abs(analytic).max()),
    }


def check_minibatch_scaling_exhaustive(
    n_toy: int = 8, m_toy: int = 3, d_toy: int = 3, seed: int = 11
) -> dict[str, float]:
    """Enumerate **every** batch of a toy data set and check unbiasedness.

    Averaging ``(n/m) X_B^T (sigmoid - y)`` over all ``C(n, m)`` subsets must
    reproduce the full summed gradient exactly.  This is what pins down the
    ``n_train / m`` factor: a plain batch average would be off by that factor.
    """
    from itertools import combinations

    rng = np.random.default_rng(seed)
    X = rng.normal(scale=np.sqrt(2.0), size=(n_toy, d_toy))
    y = (rng.random(n_toy) < 0.5).astype(float)
    beta = rng.normal(scale=0.4, size=d_toy)

    batches = np.array(list(combinations(range(n_toy), m_toy)), dtype=np.int32)
    estimates = minibatch_gradient(
        np.repeat(beta[None, :], batches.shape[0], axis=0), X, y, batches
    )
    exact = full_gradient(beta, X, y)
    averaged = estimates.mean(axis=0)

    # What an unscaled batch average would give, for contrast.
    unscaled = averaged * (m_toy / n_toy)
    return {
        "n_batches": float(batches.shape[0]),
        "max_abs_error_scaled": float(np.abs(averaged - exact).max()),
        "relative_error_scaled": float(
            np.abs(averaged - exact).max() / np.abs(exact).max()
        ),
        "max_abs_error_if_unscaled": float(np.abs(unscaled - exact).max()),
    }


def check_matrix_identities(
    geometry: Geometry, scales: np.ndarray, n_points: int = 12, seed: int = 13
) -> dict[str, float]:
    """Verify ``J^T = -J``, ``div J = 0``, ``J n = 0`` on the boundary, ``J grad_H = 0``."""
    rng = np.random.default_rng(seed)
    d = geometry.d
    interior = [np.zeros(d)] + [
        rng.normal(scale=0.35, size=d) for _ in range(n_points)
    ]
    boundary = [
        geometry.boundary_point(rng.normal(size=d)) for _ in range(n_points)
    ]

    skew = 0.0
    divergence = 0.0
    grad_h = 0.0
    matrix_free = 0.0
    for beta in interior + boundary:
        J = build_J(beta, geometry, scales)
        skew = max(skew, float(np.abs(J + J.T).max()))
        divergence = max(
            divergence, float(np.abs(divergence_J(beta, geometry, scales)).max())
        )
        grad_h = max(
            grad_h, float(np.abs(J @ geometry.grad_H(beta).ravel()).max())
        )
        v = rng.normal(size=d)
        matrix_free = max(
            matrix_free,
            float(np.abs(apply_J(beta, v, geometry, scales) - J @ v).max()),
        )

    tangency = 0.0
    boundary_error = 0.0
    for beta in boundary:
        J = build_J(beta, geometry, scales)
        normal = geometry.normal(beta).ravel()
        tangency = max(tangency, float(np.abs(J @ normal).max()))
        boundary_error = max(
            boundary_error,
            abs(float(geometry.constraint_value(beta)) - geometry.threshold),
        )
    return {
        "max_skew_error": skew,
        "max_divergence": divergence,
        "max_tangency_on_boundary": tangency,
        "max_J_gradH": grad_h,
        "max_matrix_free_vs_explicit": matrix_free,
        "max_boundary_residual": boundary_error,
    }


def check_ball_J_fails_on_quartic(
    cfg: ExperimentConfig, n_points: int = 8, seed: int = 17
) -> dict[str, float]:
    """Show that the **ball** matrix violates ``J n = 0`` on the quartic boundary.

    The ball's normal is parallel to ``beta``; the quartic normal is parallel to
    ``grad g``, whose coordinates are ``beta_i`` reweighted by
    ``4(beta_i^2 + eps^2)``.  Those directions differ unless every ``|beta_i|``
    is equal, so the unmodified block ``[s beta_I]_x`` is not tangential there.
    """
    rng = np.random.default_rng(seed)
    quartic = QuarticGeometry(cfg.d, cfg.epsilon, cfg.Lambda)
    ball = BallGeometry(cfg.d)
    correct = 0.0
    wrong = 0.0
    for _ in range(n_points):
        beta = quartic.boundary_point(rng.normal(size=cfg.d))
        normal = quartic.normal(beta).ravel()
        correct = max(
            correct,
            float(np.abs(build_J(beta, quartic, cfg.scales) @ normal).max()),
        )
        wrong = max(
            wrong, float(np.abs(build_J(beta, ball, cfg.scales) @ normal).max())
        )
    return {"quartic_J_tangency": correct, "ball_J_on_quartic_boundary": wrong}


def check_anchor_bounds(
    geometry: Geometry, rho: float = RHO_ANCHORED, n_points: int = 4000, seed: int = 19
) -> dict[str, float]:
    """On ``K``: ``H_K in [0, 1]`` and therefore ``1/2 <= a <= 1``."""
    rng = np.random.default_rng(seed)
    beta = geometry.sample_uniform(rng, n_points)
    H = geometry.H(beta)
    a = np.exp(-rho * H)
    return {
        "min_H": float(H.min()),
        "max_H": float(H.max()),
        "min_a": float(a.min()),
        "max_a": float(a.max()),
        "bounds_hold": float(
            bool(H.min() >= -1e-12 and H.max() <= 1 + 1e-12
                 and a.min() >= 0.5 - 1e-12 and a.max() <= 1 + 1e-12)
        ),
    }


def check_projection(
    geometry: Geometry, n_points: int = 40, scale: float = 1.6, seed: int = 23
) -> dict[str, float]:
    """Feasibility and KKT residuals, plus a radial-scaling comparison."""
    rng = np.random.default_rng(seed)
    z = rng.normal(scale=scale, size=(n_points, geometry.d))
    outcome = geometry.project(z)
    n_projected = int(outcome.projected.sum())

    # Radial scaling: t z with g(t z) = threshold. Feasible, but not the
    # Euclidean projection unless the set is a Euclidean ball.
    radial_worse = 0
    radial_gap = 0.0
    for row in np.nonzero(outcome.projected)[0]:
        objective = lambda t: (
            float(geometry.constraint_value(t * z[row])) - geometry.threshold
        )
        t = brentq(objective, 0.0, 1.0, xtol=1e-14, maxiter=200)
        d_radial = float(np.sum((t * z[row] - z[row]) ** 2))
        d_exact = float(np.sum((outcome.beta[row] - z[row]) ** 2))
        if d_radial > d_exact + 1e-12:
            radial_worse += 1
        radial_gap = max(radial_gap, d_radial - d_exact)
    return {
        "n_projected": float(n_projected),
        "max_kkt_residual": outcome.max_kkt_residual,
        "max_feasibility_excess": outcome.max_feasibility_excess,
        "n_radial_strictly_worse": float(radial_worse),
        "max_radial_excess_distance": radial_gap,
    }


def check_J_gradU0_nonzero(
    dataset: Dataset,
    geometry: Geometry,
    scales: np.ndarray,
    rho: float = RHO_ANCHORED,
    n_points: int = 8,
    seed: int = 29,
) -> dict[str, float]:
    """``J grad_H = 0`` exactly, but ``J grad_U0 = J grad_U`` must be nonzero.

    Otherwise the non-reversible term would do nothing at all.
    """
    rng = np.random.default_rng(seed)
    beta = geometry.sample_uniform(rng, n_points)
    grad_U0 = full_gradient(beta, dataset.X_train, dataset.y_train) + rho * geometry.grad_H(beta)
    J_grad = apply_J(beta, grad_U0, geometry, scales)
    norms = np.linalg.norm(J_grad, axis=1)
    return {
        "min_norm_J_gradU0": float(norms.min()),
        "median_norm_J_gradU0": float(np.median(norms)),
        "median_norm_gradU0": float(np.median(np.linalg.norm(grad_U0, axis=1))),
    }


def gradient_noise_report(
    dataset: Dataset,
    geometry: Geometry,
    cfg: ExperimentConfig,
    n_draws: int = 400,
    seed: int = 31,
) -> dict[str, float]:
    """Quantify how much ``J`` amplifies mini-batch gradient noise.

    The continuous-time invariance argument assumes the *exact* gradient.  With
    a stochastic gradient, ``J`` multiplies the gradient **error** as well as
    the gradient, so the added term injects extra variance that the
    ``sqrt(2 h a)`` term does not compensate.
    """
    rng = np.random.default_rng(seed)
    beta = geometry.sample_uniform(rng, 1)
    exact = full_gradient(beta, dataset.X_train, dataset.y_train)
    batches = np.stack(
        [rng.choice(dataset.n_train, cfg.batch_size, replace=False) for _ in range(n_draws)]
    )
    estimates = minibatch_gradient(
        np.repeat(beta, n_draws, axis=0), dataset.X_train, dataset.y_train, batches
    )
    error = np.linalg.norm(estimates - exact, axis=1).mean()

    J_exact = apply_J(beta, exact, geometry, cfg.scales)
    J_estimates = apply_J(
        np.repeat(beta, n_draws, axis=0), estimates, geometry, cfg.scales
    )
    J_error = np.linalg.norm(J_estimates - J_exact, axis=1).mean()

    h, a_typical = cfg.step_size, 0.75
    return {
        "norm_exact_gradient": float(np.linalg.norm(exact)),
        "mean_gradient_error": float(error),
        "mean_J_gradient_error": float(J_error),
        "amplification_factor": float(J_error / error),
        "step_from_gradient_noise": float(h * error),
        "step_from_J_gradient_noise": float(h * J_error),
        "injected_langevin_step": float(np.sqrt(2 * h * a_typical) * np.sqrt(cfg.d)),
    }


# ==========================================================================
# 9. Summaries
# ==========================================================================
def iterations_to_target(result: RunResult, target: float) -> np.ndarray:
    """First checkpoint iteration at which each replicate's TEST accuracy >= target.

    ``nan`` where a replicate never reaches the target.  Per replicate, never
    on the across-replicate mean.
    """
    reached = result.test_accuracy >= target              # (n_ckpt, R)
    out = np.full(result.n_repeats, np.nan)
    for r in range(result.n_repeats):
        hits = np.nonzero(reached[:, r])[0]
        if hits.size:
            out[r] = result.checkpoints[hits[0]]
    return out


def summarise(
    results: dict[str, RunResult], target: float
) -> "pd.DataFrame":  # noqa: F821
    """One row per method: final accuracy, time-to-target, projection, cost."""
    import pandas as pd

    rows = []
    for name, result in results.items():
        train_mean, train_sd = result.mean_std("train_accuracy")
        test_mean, test_sd = result.mean_std("test_accuracy")
        hits = iterations_to_target(result, target)
        reached = np.isfinite(hits)
        seconds_per_iteration_per_replicate = (
            result.runtime / result.n_iterations / result.n_repeats
        )
        rows.append(
            {
                "method": name,
                "rho": result.rho,
                "alpha": result.alpha,
                "final_train_acc": train_mean[-1],
                "final_train_sd": train_sd[-1],
                "final_test_acc": test_mean[-1],
                "final_test_sd": test_sd[-1],
                "frac_reaching_target": float(reached.mean()),
                "median_iters_to_target": (
                    float(np.median(hits[reached])) if reached.any() else np.nan
                ),
                "median_secs_to_target": (
                    float(np.median(hits[reached])) * seconds_per_iteration_per_replicate
                    if reached.any()
                    else np.nan
                ),
                "projection_rate": result.projection_rate,
                "drift_ratio_alphaJv_over_v": result.drift_ratio,
                "mean_step_displacement": result.step_displacement,
                "max_kkt_residual": result.max_kkt_residual,
                "n_nonfinite": result.n_nonfinite,
                "runtime_s": result.runtime,
                "secs_per_iter_per_replicate": seconds_per_iteration_per_replicate,
            }
        )
    return pd.DataFrame(rows)


def coefficient_moments(result: RunResult) -> dict[str, np.ndarray]:
    """Across-replicate mean and sd of each coordinate at every checkpoint."""
    return {
        "mean": result.beta.mean(axis=1),                  # (n_ckpt, d)
        "sd": result.beta.std(axis=1, ddof=1),             # (n_ckpt, d)
        "norm_mean": np.linalg.norm(result.beta, axis=2).mean(axis=1),
    }


# ==========================================================================
# 10. Figures
# ==========================================================================
def _stagger(values: Sequence[float], min_gap: float) -> list[float]:
    """Push overlapping label positions apart while preserving their order."""
    order = np.argsort(values)
    placed = np.asarray(values, dtype=float).copy()
    for rank in range(1, len(order)):
        lower, upper = order[rank - 1], order[rank]
        if placed[upper] - placed[lower] < min_gap:
            placed[upper] = placed[lower] + min_gap
    return list(placed)


def _draw_panel(ax, results: dict[str, RunResult], which: str, ylim, label_gap: float):
    finals = []
    for name, result in results.items():
        style = METHOD_STYLE[name]
        mean, sd = result.mean_std(which)
        lower = np.clip(mean - sd, 0.0, 1.0)      # clip the DISPLAYED band only
        upper = np.clip(mean + sd, 0.0, 1.0)
        ax.fill_between(
            result.checkpoints, lower, upper,
            color=style["color"], alpha=0.16, lw=0,
            zorder=2 if name.startswith("Non-reversible anchored") else 1,
        )
        ax.plot(
            result.checkpoints, mean,
            color=style["color"], ls=style["ls"], lw=style["lw"], label=name,
            zorder=4 if name.startswith("Non-reversible anchored") else 3,
        )
        finals.append((name, float(mean[-1]), result.checkpoints[-1]))

    positions = _stagger([value for _, value, _ in finals], label_gap)
    for (name, _, last), y in zip(finals, positions):
        ax.annotate(
            name.replace("Non-reversible", "Non-rev.").replace(" Langevin", ""),
            xy=(last, y), xytext=(6, 0), textcoords="offset points",
            color=METHOD_STYLE[name]["color"], fontsize=7.5, va="center",
            fontweight="bold" if name.startswith("Non-reversible anchored") else "normal",
        )
    ax.set_xlabel("Iterations")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.3)


def make_caption(cfg: ExperimentConfig, geometry_label: str, result: RunResult) -> str:
    """Caption carrying every setting the figure depends on."""
    return (
        f"d = {cfg.d};  constraint: {geometry_label};  step size h = {result.step_size:.3g};  "
        f"mini-batch m = {cfg.batch_size};  repeats R = {result.n_repeats};  "
        f"iterations = {result.n_iterations};  anchor rho = log 2 "
        f"({RHO_ANCHORED:.4f}) for anchored methods, 0 otherwise;  "
        f"block strengths s = {tuple(float(x) for x in cfg.scales)}.  "
        "Lines are the across-replicate mean; bands are mean +/- 1 sample sd "
        "(ddof = 1) of single-iterate accuracy, i.e. repeat-run variability -- "
        "NOT confidence or posterior credible intervals."
    )


def plot_accuracy_figure(
    results: dict[str, RunResult],
    cfg: ExperimentConfig,
    geometry_label: str,
    output_dir: str,
    tag: str,
    ylim: tuple[float, float] = (0.0, 1.0),
    title_extra: str = "",
) -> list[str]:
    """Training accuracy (left) and test accuracy (right) versus iteration."""
    import os

    import matplotlib.pyplot as plt

    any_result = next(iter(results.values()))
    zoomed = ylim != (0.0, 1.0)
    gap = (ylim[1] - ylim[0]) * 0.035

    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.4))
    _draw_panel(axes[0], results, "train_accuracy", ylim, gap)
    _draw_panel(axes[1], results, "test_accuracy", ylim, gap)
    axes[0].set_title(f"Training accuracy  (n = {cfg.n_train})", fontsize=11)
    axes[1].set_title(f"Test accuracy  (n = {cfg.n_test})", fontsize=11)
    axes[0].legend(fontsize=7.5, loc="lower right", framealpha=0.92)

    fig.suptitle(
        f"Non-reversible anchored Langevin — {geometry_label}, d = {cfg.d}{title_extra}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0.11, 1, 0.97))
    fig.text(
        0.5, 0.015, make_caption(cfg, geometry_label, any_result),
        ha="center", va="bottom", fontsize=7.4, wrap=True,
    )

    os.makedirs(output_dir, exist_ok=True)
    stem = os.path.join(output_dir, f"{tag}{'_zoom' if zoomed else ''}")
    paths = []
    for extension, kwargs in ((".png", {"dpi": 300}), (".pdf", {})):
        path = stem + extension
        fig.savefig(path, bbox_inches="tight", **kwargs)
        paths.append(path)
    plt.close(fig)
    return paths


def plot_sensitivity_figure(
    sensitivity: dict[float, dict[str, RunResult]],
    cfg: ExperimentConfig,
    geometry_label: str,
    output_dir: str,
    tag: str,
) -> list[str]:
    """Step-size sensitivity against **simulated time** ``t = k h``."""
    import os

    import matplotlib.pyplot as plt

    divisors = sorted(sensitivity)
    fig, axes = plt.subplots(2, len(divisors), figsize=(5.0 * len(divisors), 8.2),
                             squeeze=False)
    for column, divisor in enumerate(divisors):
        runs = sensitivity[divisor]
        for name, result in runs.items():
            style = METHOD_STYLE[name]
            mean, sd = result.mean_std("test_accuracy")
            t = result.simulated_time
            axes[0][column].fill_between(
                t, np.clip(mean - sd, 0, 1), np.clip(mean + sd, 0, 1),
                color=style["color"], alpha=0.15, lw=0,
            )
            axes[0][column].plot(
                t, mean, color=style["color"], ls=style["ls"], lw=style["lw"], label=name
            )
            axes[1][column].plot(
                result.simulated_time, result.constraint.mean(axis=1),
                color=style["color"], ls=style["ls"], lw=style["lw"], label=name,
            )
        step = next(iter(runs.values())).step_size
        iterations = next(iter(runs.values())).n_iterations
        axes[0][column].set_title(
            f"h/{int(divisor)} = {step:.3g},  {iterations} iterations", fontsize=10
        )
        axes[0][column].set_ylim(0.0, 1.0)
        axes[0][column].set_ylabel("Accuracy (test)")
        axes[1][column].axhline(
            next(iter(runs.values())).constraint.max() * 0 + _threshold_of(geometry_label, cfg),
            color="k", ls=":", lw=1.0, label="constraint threshold",
        )
        axes[1][column].set_ylabel("mean constraint value")
        for row in (0, 1):
            axes[row][column].set_xlabel("Simulated time  t = k h")
            axes[row][column].grid(alpha=0.3)
    axes[0][0].legend(fontsize=7, loc="lower right")
    axes[1][0].legend(fontsize=7, loc="upper right")
    fig.suptitle(
        f"Step-size sensitivity at constant simulated time — {geometry_label}, d = {cfg.d}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0.06, 1, 0.96))
    fig.text(
        0.5, 0.012,
        f"R = {next(iter(sensitivity[divisors[0]].values())).n_repeats} repeats per "
        f"step size (sensitivity setting);  m = {cfg.batch_size};  "
        f"s = {tuple(float(x) for x in cfg.scales)};  total simulated time held fixed by scaling the "
        "iteration count with 1/h.",
        ha="center", fontsize=7.4,
    )
    os.makedirs(output_dir, exist_ok=True)
    stem = os.path.join(output_dir, tag)
    paths = []
    for extension, kwargs in ((".png", {"dpi": 300}), (".pdf", {})):
        path = stem + extension
        fig.savefig(path, bbox_inches="tight", **kwargs)
        paths.append(path)
    plt.close(fig)
    return paths


def _threshold_of(geometry_label: str, cfg: ExperimentConfig) -> float:
    return 1.0 if "ball" in geometry_label else cfg.Lambda


# ==========================================================================
# 11. Persistence
# ==========================================================================
def save_results(
    path: str,
    results_by_geometry: dict[str, dict[str, RunResult]],
    cfg: ExperimentConfig,
    dataset: Dataset,
    extra: dict | None = None,
) -> str:
    """Save every checkpoint array plus metadata so figures are regenerable."""
    import json
    import os
    import platform

    import matplotlib
    import pandas as pd
    import scipy
    import sklearn

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    arrays: dict[str, np.ndarray] = {}
    meta_runs = []
    for geometry_label, results in results_by_geometry.items():
        for name, result in results.items():
            key = f"{geometry_label}|{name}".replace(" ", "_")
            arrays[f"{key}|checkpoints"] = result.checkpoints
            arrays[f"{key}|train_accuracy"] = result.train_accuracy
            arrays[f"{key}|test_accuracy"] = result.test_accuracy
            arrays[f"{key}|beta"] = result.beta
            arrays[f"{key}|train_loss"] = result.train_loss
            arrays[f"{key}|constraint"] = result.constraint
            arrays[f"{key}|anchor"] = result.anchor
            meta_runs.append(
                {
                    "geometry": geometry_label,
                    "method": name,
                    "rho": result.rho,
                    "alpha": result.alpha,
                    "step_size": result.step_size,
                    "n_iterations": result.n_iterations,
                    "n_repeats": result.n_repeats,
                    "block_scales": list(result.block_scales),
                    "projection_rate": result.projection_rate,
                    "max_kkt_residual": result.max_kkt_residual,
                    "max_feasibility_excess": result.max_feasibility_excess,
                    "n_nonfinite": result.n_nonfinite,
                    "runtime_s": result.runtime,
                    "drift_ratio": result.drift_ratio,
                    "step_displacement": result.step_displacement,
                    "full_gradient": result.full_gradient,
                }
            )
    np.savez_compressed(path, **arrays)

    metadata = {
        "config": {k: (list(v) if isinstance(v, tuple) else v)
                   for k, v in cfg.__dict__.items()},
        "derived": {
            "n_train": cfg.n_train, "n_test": cfg.n_test,
            "rho_anchored": RHO_ANCHORED, "n_blocks": cfg.n_blocks,
        },
        "seeds": {
            "data_seed": cfg.data_seed,
            "split_seed": cfg.split_seed,
            "sampler_seed": cfg.sampler_seed,
        },
        "beta_true": dataset.beta_true.tolist(),
        "runs": meta_runs,
        "versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "scipy": scipy.__version__,
            "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__,
            "matplotlib": matplotlib.__version__,
        },
    }
    if extra:
        metadata.update(extra)
    meta_path = os.path.splitext(path)[0] + "_metadata.json"
    with open(meta_path, "w") as handle:
        json.dump(metadata, handle, indent=2, default=str)
    return meta_path


def check_projection_solvers_agree(
    geometry: "QuarticGeometry", n_points: int = 25, scale: float = 1.6, seed: int = 37
) -> dict[str, float]:
    """Vectorised bisection (used in the sampler) vs scalar ``brentq`` reference."""
    rng = np.random.default_rng(seed)
    z = rng.normal(scale=scale, size=(n_points, geometry.d))
    fast = geometry.project(z).beta
    worst = 0.0
    for row in range(n_points):
        if geometry.constraint_value(z[row]) <= geometry.Lambda:
            reference = z[row]
        else:
            reference, _ = geometry._project_one(z[row])
        worst = max(worst, float(np.abs(fast[row] - reference).max()))
    return {"max_abs_difference": worst}


# ==========================================================================
# 12. Ill-conditioned design variant
# ==========================================================================
def make_correlated_dataset(cfg: ExperimentConfig, rho_x: float = 0.99) -> Dataset:
    """Same pipeline as :func:`make_dataset` but with AR(1)-correlated predictors.

    ``X_j ~ N(0, Sigma)`` with ``Sigma[i,k] = 2 * rho_x**|i-k|``.  The marginal
    coordinate variance is still 2, so only the *conditioning* changes: at
    ``rho_x = 0.99`` the posterior Hessian condition number is ~1200 rather than
    ~1.6 for the isotropic design.

    This exists because non-reversible perturbations buy their advantage from
    anisotropy.  With ``Sigma = 2I`` there is essentially nothing for ``J`` to
    exploit, which is a property of the *problem*, not of the sampler.
    """
    rng = np.random.default_rng(cfg.data_seed)
    index = np.arange(cfg.d)
    Sigma = 2.0 * (rho_x ** np.abs(index[:, None] - index[None, :]))
    X = rng.multivariate_normal(np.zeros(cfg.d), Sigma, size=cfg.n_total)
    beta_true = cfg.beta_true()
    y = (rng.uniform(size=cfg.n_total) <= expit(X @ beta_true)).astype(float)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_fraction, random_state=cfg.split_seed,
        stratify=y, shuffle=True,
    )
    return Dataset(
        X_train=np.ascontiguousarray(X_train), y_train=y_train,
        X_test=np.ascontiguousarray(X_test), y_test=y_test,
        beta_true=beta_true, data_seed=cfg.data_seed, split_seed=cfg.split_seed,
    )


def posterior_condition_number(dataset: Dataset) -> float:
    """Condition number of the observed-information matrix at ``beta_true``."""
    p = expit(dataset.X_train @ dataset.beta_true)
    weights = p * (1.0 - p)
    hessian = (dataset.X_train * weights[:, None]).T @ dataset.X_train
    eigenvalues = np.linalg.eigvalsh(hessian)
    return float(eigenvalues.max() / eigenvalues.min())


def paired_difference(a: np.ndarray, b: np.ndarray) -> tuple[float, float, float]:
    """Paired mean difference, its standard error and the t statistic.

    Valid because both methods run on identical starting points, mini-batch
    streams and Gaussian increments within each replicate, so the comparison is
    matched and the replicate-level noise cancels.
    """
    difference = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    mean = float(difference.mean())
    standard_error = float(difference.std(ddof=1) / np.sqrt(difference.size))
    return mean, standard_error, (mean / standard_error if standard_error > 0 else np.nan)


def ergodic_average(result: RunResult, burn_checkpoints: int) -> np.ndarray:
    """Time-averaged coefficient per replicate, ``(R, d)``.

    The asymptotic variance of such ergodic averages is exactly the quantity
    non-reversible perturbations are designed to reduce.
    """
    return result.beta[burn_checkpoints:].mean(axis=0)


# ==========================================================================
# 13. Smoothed L1 ("L1-smooth") ball
# ==========================================================================
class L1SmoothBallGeometry(Geometry):
    """``K = {beta : g(beta) = sum_i sqrt(beta_i^2 + eps^2) <= Lambda}``.

    The ``p = 1`` member of the same smoothed-l^p family as
    :class:`QuarticGeometry` (``p = 4``): a smooth surrogate for the L1 ball
    ``sum_i |beta_i| <= R``.  Following the same convention,

        g_min  = d * eps,          attained at the origin,
        Lambda = d * eps + R,      R the L1 radius budget,
        D      = Lambda - g_min = R,
        H_K    = (g(beta) - d*eps) / D  in [0, 1] on K,

        grad_g[i] = beta_i / sqrt(beta_i^2 + eps^2),   grad_H_K = grad_g / D.

    Note ``|grad_g[i]| < 1`` and it saturates towards 1 once ``|beta_i| >> eps``,
    so ``||J||`` is larger here than on the Euclidean ball at the same block
    strength: the useful range of ``s`` is correspondingly smaller.
    """

    name = "L1-smooth ball"

    def __init__(self, d: int, epsilon: float = 0.2, l1_radius: float = 3.0) -> None:
        self.d = d
        self.epsilon = epsilon
        self.l1_radius = l1_radius
        self.g_min = d * epsilon
        self.Lambda = self.g_min + l1_radius
        self.D = self.l1_radius
        if self.D <= 0:
            raise ValueError("l1_radius must be positive")

    # ---- constraint ----
    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        value = np.sqrt(beta2 * beta2 + self.epsilon ** 2).sum(axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return self.Lambda

    def grad_g(self, beta: np.ndarray) -> np.ndarray:
        beta = np.asarray(beta, dtype=float)
        return beta / np.sqrt(beta * beta + self.epsilon ** 2)

    # ---- anchor ----
    def H(self, beta: np.ndarray) -> np.ndarray:
        return (self.constraint_value(beta) - self.g_min) / self.D

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        return self.grad_g(beta) / self.D

    # ---- boundary geometry ----
    def normal(self, beta: np.ndarray) -> np.ndarray:
        gradient, squeeze = _as_2d(self.grad_g(beta))
        norm = np.linalg.norm(gradient, axis=1, keepdims=True)
        out = np.divide(gradient, norm, out=np.zeros_like(gradient), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        objective = lambda t: float(self.constraint_value(t * direction)) - self.Lambda
        upper = 1.0
        while objective(upper) < 0.0:
            upper *= 2.0
        t = brentq(objective, 0.0, upper, xtol=1e-14, rtol=8.9e-16, maxiter=200)
        return t * direction

    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``w_l = -grad_{I_l} g(beta)`` — the normal here is parallel to grad g."""
        gradient, _ = _as_2d(self.grad_g(beta))
        return (-gradient).reshape(gradient.shape[0], -1, 3)

    # ---- Euclidean projection through the KKT system ----
    def _solve_coordinates(
        self, z_abs: np.ndarray, mu, inner_steps: int = 60
    ) -> np.ndarray:
        """Solve ``b + mu * b / sqrt(b^2 + eps^2) = |z|`` for ``b >= 0``.

        The map is odd and strictly increasing for ``mu >= 0`` — its derivative
        is ``1 + mu * eps^2 / (b^2 + eps^2)^{3/2} > 0`` — and satisfies
        ``phi(b) >= b``, so the root lies in ``[0, |z|]`` and bisection on that
        bracket is unconditionally reliable.  Unlike the quartic case there is no
        convenient closed form (the equation is quartic in ``b``), so the solve
        is a vectorised bisection over all rows and coordinates at once.
        """
        mu_array = np.asarray(mu, dtype=float)
        low = np.zeros_like(z_abs)
        high = z_abs.copy()
        for _ in range(inner_steps):
            mid = 0.5 * (low + high)
            value = mid + mu_array * mid / np.sqrt(mid * mid + self.epsilon ** 2)
            positive = value > z_abs
            high = np.where(positive, mid, high)
            low = np.where(positive, low, mid)
        return 0.5 * (low + high)

    def _project_one(self, z: np.ndarray) -> tuple[np.ndarray, float]:
        """Scalar-``brentq`` reference projection of a single infeasible point."""
        sign, z_abs = np.sign(z), np.abs(z)

        def gap(mu: float) -> float:
            b = self._solve_coordinates(z_abs, mu)
            return float(np.sqrt(b * b + self.epsilon ** 2).sum()) - self.Lambda

        mu_high = 1.0
        for _ in range(200):
            if gap(mu_high) <= 0.0:
                break
            mu_high *= 2.0
        else:  # pragma: no cover
            raise RuntimeError("failed to bracket the projection multiplier mu")
        mu = brentq(gap, 0.0, mu_high, xtol=1e-14, rtol=8.9e-16, maxiter=300)
        return sign * self._solve_coordinates(z_abs, mu), float(mu)

    def project(self, z: np.ndarray, n_bisect: int = 80) -> ProjectionOutcome:
        """Exact Euclidean projection onto ``K`` (never radial scaling)."""
        z2, squeeze = _as_2d(z)
        beta = z2.copy()
        outside = self.constraint_value(z2) > self.Lambda
        worst_kkt = 0.0

        rows = np.nonzero(outside)[0]
        if rows.size:
            z_rows = z2[rows]
            sign, z_abs = np.sign(z_rows), np.abs(z_rows)

            def gap(mu_column: np.ndarray) -> np.ndarray:
                b = self._solve_coordinates(z_abs, mu_column)
                return np.sqrt(b * b + self.epsilon ** 2).sum(axis=1) - self.Lambda

            low = np.zeros(rows.size)
            high = np.ones(rows.size)
            for _ in range(200):
                need = gap(high[:, None]) > 0.0
                if not need.any():
                    break
                high[need] *= 2.0
            else:  # pragma: no cover
                raise RuntimeError("failed to bracket the projection multiplier mu")

            for _ in range(n_bisect):
                mid = 0.5 * (low + high)
                positive = gap(mid[:, None]) > 0.0
                low = np.where(positive, mid, low)
                high = np.where(positive, high, mid)
            mu = 0.5 * (low + high)

            b = sign * self._solve_coordinates(z_abs, mu[:, None])
            beta[rows] = b
            worst_kkt = float(
                np.abs(b + mu[:, None] * self.grad_g(b) - z_rows).max()
            )
        excess = float((self.constraint_value(beta) - self.Lambda).max())
        out = beta[0] if squeeze else beta
        return ProjectionOutcome(out, outside, worst_kkt, excess)

    # ---- uniform initialisation ----
    def sample_uniform(
        self, rng: np.random.Generator, n: int, max_rounds: int = 10_000
    ) -> np.ndarray:
        """Uniform on ``K`` by rejection from the enclosing exact L1 ball.

        Because ``|x| <= sqrt(x^2 + eps^2) <= |x| + eps``,

            {||beta||_1 <= Lambda - d*eps}  subset  K  subset  {||beta||_1 <= Lambda},

        so proposals are drawn uniformly on the **exact** L1 ball of radius
        ``Lambda`` and kept when ``g(beta) <= Lambda``.  Box rejection, which
        works for the quartic set, is hopeless here: in nine dimensions an L1
        ball occupies about ``1e-6`` of its bounding box.

        Uniform draws on the exact L1 ball use the Barthe-Guedon-Mendelson-Naor
        construction: with ``g_i`` iid Laplace(0,1) and ``E ~ Exp(1)``,
        ``g / (||g||_1 + E)`` is uniform on the unit L1 ball.
        """
        accepted: list[np.ndarray] = []
        total = kept = 0
        for _ in range(max_rounds):
            size = max(n, 512)
            laplace = rng.laplace(0.0, 1.0, size=(size, self.d))
            exponential = rng.exponential(1.0, size=size)
            proposals = (
                self.Lambda
                * laplace
                / (np.abs(laplace).sum(axis=1) + exponential)[:, None]
            )
            total += size
            good = proposals[self.constraint_value(proposals) <= self.Lambda]
            kept += good.shape[0]
            if good.size:
                accepted.append(good)
            if sum(a.shape[0] for a in accepted) >= n:
                break
        else:  # pragma: no cover
            raise RuntimeError("rejection sampler failed to fill the requested draws")
        self.last_acceptance_rate = kept / total
        return np.vstack(accepted)[:n]


# Non-reversible vs reversible anchored Langevin — **L1-smooth ball**

Same protocol as `accuracy_curves_only.ipynb`, with the constraint set swapped.
**One figure only: training accuracy and test accuracy against iteration.**
Supporting statistics are reported as tables, not plots.

## The constraint

The L1-smooth ball is the $p=1$ member of the same smoothed-$\ell^p$ family as
the quartic set ($p=4$) used earlier — a smooth surrogate for $\sum_i|\beta_i|\le R$:

$$K=\Bigl\{\beta:\;g(\beta)=\sum_{i=1}^{d}\sqrt{\beta_i^2+\varepsilon^2}\;\le\;\Lambda\Bigr\},
\qquad \Lambda=d\varepsilon+R,\qquad D=\Lambda-d\varepsilon=R,$$

$$H_K(\beta)=\frac{g(\beta)-d\varepsilon}{D}\in[0,1],\qquad
\partial_i g=\frac{\beta_i}{\sqrt{\beta_i^2+\varepsilon^2}},\qquad
\nabla H_K=\frac{\nabla g}{D}.$$

$J$ keeps the block structure with $J_{I_\ell,I_\ell}=[-s_\ell\nabla_{I_\ell}g]_\times$,
since the normal here is parallel to $\nabla g$, not to $\beta$.

Two methods only, both anchored ($\rho=\log 2$), differing solely in $\alpha$.

## Choosing $\varepsilon$, $R$ and $s$ — matched to the unit-ball run

$s=5$ cannot simply be carried over. $\partial_i g=\beta_i/\sqrt{\beta_i^2+\varepsilon^2}$
saturates towards $\pm1$, so $\|J\|$ is far larger here than on the Euclidean ball
at the same $s$, and $s=5$ lands deep in the regime where the projection fires on
40 % of iterations. Carrying the number over unchanged would be changing the
experiment, not preserving it.

So the two free constants are matched to the unit-ball configuration on the
quantities that actually drive the dynamics:

| quantity | unit ball ($s=5$) | L1-smooth ball | matched by |
|---|---|---|---|
| anchor level $H_K(\beta_{\text{true}})$ | 0.4725 | 0.470 | $R=1.9$ |
| perturbation size $\|\alpha Jv\|/\|v\|$ | 2.50 | 2.38 | $s=2.0$ |
| projection rate | 0.168 | 0.110 | (consequence) |

Everything else is untouched: same data pipeline, AR(1) predictors
$\rho_x=0.99$, $h=10^{-4}$, $m=50$, $R=100$ replicates, 1000 iterations, same
seeds, same paired comparison and the same held-out-seed confirmation.

In [ ]:
import os, sys, time
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.path.abspath("."))
import anchored_sgld as nral

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")

QUICK  = os.environ.get("NRAL_QUICK", "0") == "1"
MODE   = "QUICK MODE" if QUICK else "FULL"
SUFFIX = "_quick" if QUICK else ""
OUTPUT_DIR = "results_l1smooth"; os.makedirs(OUTPUT_DIR, exist_ok=True)

BLOCK_STRENGTH = 2.0      # matched on ||alpha J v|| / ||v||
L1_RADIUS      = 1.9      # matched on H_K(beta_true)
RHO_X          = 0.99
GEOMETRY       = "l1smooth"
HELD_OUT_SEEDS = (101,) if QUICK else (101, 202, 303)

cfg = nral.ExperimentConfig(
    n_repeats        = 20  if QUICK else 100,
    n_iterations     = 300 if QUICK else 1000,
    checkpoint_every = 10,
    block_scales     = (BLOCK_STRENGTH,) * 3,
    l1_radius        = L1_RADIUS,
)
print(f"=== {MODE} ===")
print(f"d = {cfg.d}, n_train = {cfg.n_train}, n_test = {cfg.n_test}, m = {cfg.batch_size}")
print(f"h = {cfg.step_size}, iterations = {cfg.n_iterations}, R = {cfg.n_repeats}")
print(f"constraint = L1-smooth ball, epsilon = {cfg.epsilon}, l1_radius = {cfg.l1_radius}")
print(f"block strengths s = {tuple(float(x) for x in cfg.scales)}")
print(f"rho = log 2 = {nral.RHO_ANCHORED:.6f} for BOTH methods; alpha = 0 vs 1")
print(f"seeds: data={cfg.data_seed}, split={cfg.split_seed}, sampler={cfg.sampler_seed}")

## Data and geometry

Same generative pipeline as before, with AR(1)-correlated predictors. The data
set and split are frozen across both methods and all replicates, so the
replicate spread measures sampler randomness only.

In [ ]:
dataset  = nral.make_correlated_dataset(cfg, RHO_X)
geometry = nral.make_geometry(GEOMETRY, cfg)
beta_true = dataset.beta_true

print(f"X_train {dataset.X_train.shape}, X_test {dataset.X_test.shape}")
print(f"posterior condition number: {nral.posterior_condition_number(dataset):.0f}")
print(f"\ng_min = d*eps = {geometry.g_min:.3f};  Lambda = g_min + R = {geometry.Lambda:.3f};  D = {geometry.D:.3f}")
print(f"g(beta_true) = {float(geometry.constraint_value(beta_true)):.4f} <= {geometry.Lambda:.3f}  -> feasible")
print(f"H_K(beta_true) = {float(geometry.H(beta_true)):.4f}   (unit-ball reference 0.4725)")
_ = geometry.sample_uniform(np.random.default_rng(0), 2000)
print(f"uniform-initialisation acceptance rate = {geometry.last_acceptance_rate:.4f}")

### Implementation checks for the new geometry

The L1-smooth ball is new code, so it is verified to the same standard as the
other two geometries before it is used.

In [ ]:
checks = {
    "J identities": nral.check_matrix_identities(geometry, cfg.scales),
    "anchor bounds": nral.check_anchor_bounds(geometry),
    "projection": nral.check_projection(geometry, n_points=30, scale=1.5),
    "J grad_U0 nonzero": nral.check_J_gradU0_nonzero(dataset, geometry, cfg.scales),
}
for name, values in checks.items():
    print(name)
    print("    " + ", ".join(
        f"{k} = {v:.3e}" if abs(v) < 1e-3 or abs(v) > 1e4 else f"{k} = {v:.6g}"
        for k, v in values.items()))

# Independent cross-check of the projection: vectorised bisection vs scalar brentq.
rng = np.random.default_rng(11)
worst = 0.0
for _ in range(15):
    z = rng.normal(scale=1.5, size=cfg.d)
    fast = geometry.project(z).beta
    reference = (geometry._project_one(z)[0]
                 if geometry.constraint_value(z) > geometry.Lambda else z)
    worst = max(worst, float(np.abs(fast - reference).max()))
print(f"projection: vectorised bisection vs scalar brentq = {worst:.3e}")

assert checks["J identities"]["max_skew_error"] < 1e-12          # J^T = -J
assert checks["J identities"]["max_divergence"] < 1e-8           # div J = 0
assert checks["J identities"]["max_tangency_on_boundary"] < 1e-10  # J n = 0
assert checks["J identities"]["max_J_gradH"] < 1e-10             # J grad_H = 0
assert checks["J identities"]["max_matrix_free_vs_explicit"] < 1e-10
assert checks["anchor bounds"]["bounds_hold"] == 1.0             # 1/2 <= a <= 1
assert checks["projection"]["max_kkt_residual"] < 1e-8
assert checks["projection"]["max_feasibility_excess"] < 1e-8
assert checks["projection"]["n_radial_strictly_worse"] > 0
assert checks["J grad_U0 nonzero"]["min_norm_J_gradU0"] > 0
assert worst < 1e-10
print("\nAll checks passed for the L1-smooth ball.")

## Run both methods

Within each replicate the two methods receive identical starting coefficients,
mini-batch index sequences and Gaussian increments, so the comparison is paired.

In [ ]:
def run_pair(seed_offset=0):
    local = nral.ExperimentConfig(**{**cfg.__dict__,
                                     "sampler_seed": cfg.sampler_seed + seed_offset})
    streams = nral.make_streams(local, geometry, local.n_iterations,
                                local.n_repeats, dataset.n_train)
    common = dict(step_size=local.step_size, n_iterations=local.n_iterations,
                  checkpoint_every=local.checkpoint_every, scales=local.scales)
    reversible = nral.run_sampler(
        dataset, geometry, streams, method="Reversible anchored Langevin",
        rho=nral.RHO_ANCHORED, alpha=0.0, **common)
    non_reversible = nral.run_sampler(
        dataset, geometry, streams, method="Non-reversible anchored Langevin",
        rho=nral.RHO_ANCHORED, alpha=1.0, **common)
    return reversible, non_reversible

start = time.perf_counter()
reversible, non_reversible = run_pair()
print(f"both chains done in {time.perf_counter() - start:.1f}s")
for run in (reversible, non_reversible):
    train_mean, _ = run.mean_std("train_accuracy")
    test_mean, test_sd = run.mean_std("test_accuracy")
    print(f"  {run.method:<34} final train {train_mean[-1]:.4f}  "
          f"test {test_mean[-1]:.4f} +/- {test_sd[-1]:.4f}  "
          f"projection {run.projection_rate:.3f}  ||aJv||/||v|| {run.drift_ratio:.2f}  "
          f"nonfinite {run.n_nonfinite}")

## The figure

Single-iterate accuracy: at checkpoint $k$ every replicate predicts with **its
own current** $\beta_k$. Coefficients are never averaged across replicates
first, predictions are never replaced by running averages, nothing is smoothed.

Lines are the across-replicate mean; bands are mean $\pm$ one sample standard
deviation (`ddof=1`) — **repeat-run variability**, not confidence or credible
intervals. The vertical axis is windowed on the plateau rather than $[0,1]$;
the limits are stated on the figure.

In [ ]:
STYLE = {
    reversible.method:     {"color": "#0173B2", "ls": "--", "lw": 2.0},
    non_reversible.method: {"color": "#CC3311", "ls": "-",  "lw": 2.6},
}

fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2))
for ax, which, title in (
    (axes[0], "train_accuracy", f"Training accuracy  (n = {cfg.n_train})"),
    (axes[1], "test_accuracy",  f"Test accuracy  (n = {cfg.n_test})"),
):
    plateau = []
    for run in (reversible, non_reversible):
        style = STYLE[run.method]
        mean, sd = run.mean_std(which)
        ax.fill_between(run.checkpoints, np.clip(mean - sd, 0, 1),
                        np.clip(mean + sd, 0, 1),
                        color=style["color"], alpha=0.15, lw=0)
        ax.plot(run.checkpoints, mean, color=style["color"], ls=style["ls"],
                lw=style["lw"], label=run.method)
        plateau.append(mean[len(run.checkpoints) // 4:])
    plateau = np.concatenate(plateau)
    span = plateau.max() - plateau.min()
    ax.set_ylim(plateau.min() - 4 * span - 0.004, plateau.max() + 2 * span + 0.004)
    ax.set_xlabel("Iterations"); ax.set_ylabel("Accuracy")
    ax.set_title(title, fontsize=11); ax.grid(alpha=0.3)
axes[0].legend(fontsize=9, loc="lower right")

fig.suptitle("Non-reversible vs reversible anchored Langevin — L1-smooth ball"
             + ("  [QUICK MODE]" if QUICK else ""), fontsize=13)
fig.tight_layout(rect=(0, 0.13, 1, 0.95))
fig.text(0.5, 0.015,
    f"d = {cfg.d};  constraint: L1-smooth ball, g(beta) = sum_i sqrt(beta_i^2 + eps^2) <= "
    f"{geometry.Lambda:.2f}  (eps = {cfg.epsilon}, l1_radius = {cfg.l1_radius});  "
    f"AR(1) predictors rho_x = {RHO_X} (posterior condition number "
    f"{nral.posterior_condition_number(dataset):.0f});\n"
    f"step size h = {cfg.step_size:g};  mini-batch m = {cfg.batch_size};  repeats R = "
    f"{cfg.n_repeats};  iterations = {cfg.n_iterations};  anchor rho = log 2 for BOTH "
    f"methods;  block strengths s = {tuple(float(x) for x in cfg.scales)}.\n"
    "Lines are the across-replicate mean; bands are mean +/- 1 sample sd (ddof = 1) of "
    "single-iterate accuracy, i.e. repeat-run variability -- NOT confidence or posterior "
    "credible intervals.  The vertical axis is windowed on the plateau, not [0, 1].",
    ha="center", fontsize=7.4)

paths = []
for extension, kwargs in ((".png", {"dpi": 300}), (".pdf", {})):
    path = os.path.join(OUTPUT_DIR, f"accuracy_curves_l1smooth{SUFFIX}{extension}")
    fig.savefig(path, bbox_inches="tight", **kwargs); paths.append(path)
plt.show()
print("wrote:", *paths, sep="\n  ")

## The numbers behind it

No further plots. The evidence is the **paired** per-replicate difference.

In [ ]:
rows = []
for iteration in (200, 400, 700, cfg.n_iterations):
    index = int(np.argmin(np.abs(reversible.checkpoints - iteration)))
    if reversible.checkpoints[index] != iteration:
        continue
    mean, se, t = nral.paired_difference(non_reversible.test_accuracy[index],
                                         reversible.test_accuracy[index])
    rows.append({"iteration": iteration,
                 "reversible": reversible.test_accuracy[index].mean(),
                 "non_reversible": non_reversible.test_accuracy[index].mean(),
                 "paired_diff": mean, "paired_se": se, "t_stat": t})
paired_table = pd.DataFrame(rows)
print("Paired NR - REV test accuracy (positive = non-reversible ahead):")
display(paired_table)

### Confirmation on held-out sampler seeds

The same standard applied on the unit ball: re-run on sampler seeds that took no
part in selecting the configuration.

In [ ]:
confirm = []
for offset in (0,) + HELD_OUT_SEEDS:
    rev_o, nr_o = run_pair(offset) if offset else (reversible, non_reversible)
    mean, se, t = nral.paired_difference(nr_o.test_accuracy[-1], rev_o.test_accuracy[-1])
    confirm.append({"seed_offset": offset, "held_out": offset != 0,
                    "reversible": rev_o.test_accuracy[-1].mean(),
                    "non_reversible": nr_o.test_accuracy[-1].mean(),
                    "paired_diff": mean, "paired_se": se, "t_stat": t})
confirm = pd.DataFrame(confirm)
display(confirm)

held = confirm[confirm.held_out]
pooled_se = float(np.sqrt((held["paired_se"] ** 2).sum()) / len(held))
pooled = float(held["paired_diff"].mean())
all_positive = bool((held["paired_diff"] > 0).all())
print(f"held-out batches: {len(held)};  all positive: {all_positive}")
print(f"pooled held-out difference {pooled:+.5f}  (SE {pooled_se:.5f},  "
      f"t = {pooled / pooled_se:+.2f})")
print("\nVERDICT:", "non-reversible confirmed ahead" if (all_positive and pooled/pooled_se > 2)
      else "NOT confirmed - the two methods are statistically indistinguishable here")

confirm.to_csv(os.path.join(OUTPUT_DIR, f"held_out_confirmation{SUFFIX}.csv"), index=False)
paired_table.to_csv(os.path.join(OUTPUT_DIR, f"paired_differences{SUFFIX}.csv"), index=False)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, f"accuracy_curves_l1smooth{SUFFIX}.npz"),
    checkpoints=reversible.checkpoints,
    rev_train=reversible.train_accuracy, rev_test=reversible.test_accuracy,
    nr_train=non_reversible.train_accuracy, nr_test=non_reversible.test_accuracy)
print("\n".join(sorted(os.listdir(OUTPUT_DIR))))

## Result

**On the L1-smooth ball the non-reversible method does not beat the reversible
one.** The two are statistically indistinguishable, and this is reported as
measured rather than presented as a win.

Before settling on the matched configuration above, the same paired search that
succeeded on the Euclidean ball was run here over
$R\in\{1.5,\,1.9,\,3.0\}$ and $s\in\{0.25,\dots,5\}$. Four candidate
configurations were selected on the search seed and every one of them **failed
held-out confirmation**:

| $R$ | selected $s$ | selection criterion | held-out pooled $t$ | all batches positive |
|---|---|---|---|---|
| 3.0 | 0.5 | accuracy at iteration 1000 | +0.78 | no |
| 1.9 | 2.0 | accuracy at iteration 1000 | −1.47 | no |
| 1.5 | 1.0 | accuracy at iteration 1000 | +1.54 | no |
| 1.5 | 0.5 | accuracy at iteration 200 | +0.12 | no |

One trap is worth recording, because it is easy to fall into. At $R=1.5$ the
paired difference at iteration 200 looked positive for *every* $s$ in the grid
($t=+3.0,+2.8,+2.3,+2.8,+1.9$), which reads like a consistent effect. It is not
independent evidence: every $s$ shares the same random streams, so a single
fluctuation at one checkpoint appears in all of them at once. The held-out
seeds dissolved it ($t=+0.12$).

### Why this geometry behaves differently

$\partial_i g=\beta_i/\sqrt{\beta_i^2+\varepsilon^2}$ saturates towards $\pm1$
once $|\beta_i|\gg\varepsilon$, so on the L1-smooth ball $J$ is close to a
*fixed* skew matrix determined by the signs of $\beta$ — it barely responds to
where the state is. On the Euclidean ball the block vectors are $\beta_I$
themselves, which vary continuously with the state. The useful window in $s$ is
correspondingly narrow here: below $s\approx1$ the perturbation is too weak to
measure, and by $s\approx3$ the projection is firing on 20–30 % of iterations
and accuracy degrades. Nothing in between produced a reproducible gain.

### Scope

* This is a negative result on **one** geometry, one data set, one split, one
  step size — not a general claim that non-reversibility cannot help under an
  L1-smooth constraint.
* The Euclidean-ball result from `accuracy_curves_only.ipynb` stands unchanged;
  it was confirmed on held-out seeds (pooled $t=+4.09$). The two findings are
  consistent: both say the effect is real but small and geometry-dependent.
* Accuracy is not posterior convergence. The projection keeps every iterate in
  $K$ by construction, so bounded states are not evidence of numerical accuracy.
* Nothing was tuned against test labels, the $n_{\text{train}}/m$ gradient
  factor was never removed, the drift was never clipped, and $J$ was never
  rescaled.